# APS360 — Chip Congestion Prediction: data check

Goal of this notebook: confirm the CircuitNet-N28 congestion data is reachable from Colab, inspect it, and build a small preprocessed subset.

**Before running:**
1. Open https://circuitnet.github.io/intro/download.html and follow the **CircuitNet-N28 → Google Drive** link.
2. Inside the shared folder, find the routability features. You only need these sub-folders (not the full ~200 GB):
   - `macro_region`
   - `RUDY` and `RUDY_pin` (may sit under a `RUDY/` parent)
   - `congestion` → the **GR horizontal overflow** and **GR vertical overflow** labels
3. Right-click each needed folder → **Organize → Add shortcut → My Drive** → put them in a folder called `CircuitNet_N28`.
4. Run the cells below in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, subprocess
SRC = '/content/drive/MyDrive/CircuitNet_N28'   # where you put the shortcuts
RAW = '/content/raw'                              # decompressed data (local Colab disk, faster)
OUT = '/content/drive/MyDrive/aps360_congestion'  # preprocessed .npy, saved to Drive so it survives restarts
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)

# 1) What did we actually get? Sizes tell us whether this fits on Colab (~100 GB disk).
!du -sh {SRC}/* 2>/dev/null
!find {SRC} -maxdepth 3 | head -50
!df -h /content | tail -1

### Decompress
The Drive copies are split/compressed archives. The official `decompress_routability.py` (in the same Drive folder) handles this; the cell below runs it if present, otherwise falls back to a generic untar.
**Look at the output of the previous cell first** — if names differ, adjust the paths.

In [ ]:
script = glob.glob(f'{SRC}/**/decompress_routability.py', recursive=True)
print('official script:', script)

def untar_all(folder, dest):
    # handles split archives like name.tar.gz.aa, .ab ... and plain .tar.gz
    parts = sorted(glob.glob(f'{folder}/*.tar.gz*'))
    groups = {}
    for p in parts:
        key = p.split('.tar.gz')[0]
        groups.setdefault(key, []).append(p)
    for key, ps in groups.items():
        print('extracting', os.path.basename(key), f'({len(ps)} parts)')
        subprocess.run(f'cat {" ".join(map(repr, ps))} | tar -xz -C {dest}', shell=True, check=True)

for d in sorted(glob.glob(f'{SRC}/*')):
    if os.path.isdir(d):
        untar_all(d, RAW)

!du -sh {RAW}/* 2>/dev/null
!find {RAW} -maxdepth 2 -type d | head -40

### Inspect one sample
Set the four folder paths from the listing above.

In [ ]:
import numpy as np, re, matplotlib.pyplot as plt
from collections import Counter

# >>> EDIT these to match the directory listing above <<<
F_MACRO = glob.glob(f'{RAW}/**/macro_region', recursive=True)[0]
F_RUDY  = [p for p in glob.glob(f'{RAW}/**/RUDY', recursive=True) if os.path.isdir(p)][0]
F_RPIN  = glob.glob(f'{RAW}/**/RUDY_pin', recursive=True)[0]
F_LH    = glob.glob(f'{RAW}/**/*GR_horizontal_overflow*', recursive=True)[0]
F_LV    = glob.glob(f'{RAW}/**/*GR_vertical_overflow*', recursive=True)[0]
print(F_MACRO, F_RUDY, F_RPIN, F_LH, F_LV, sep='\n')

names = sorted(os.listdir(F_MACRO))
print(len(names), 'samples; example names:', names[:3])

# Design id is embedded in the file name -> needed for the design-held-out split
DESIGN_RE = re.compile(r'(RISCY-FPU-[ab]|RISCY-[ab]|zero-riscy-[ab])')
designs = [DESIGN_RE.search(n).group(1) if DESIGN_RE.search(n) else 'UNKNOWN' for n in names]
print(Counter(designs))

def load(folder, n):
    a = np.load(os.path.join(folder, n), allow_pickle=True)
    return a.astype(np.float32)

n = names[0]
maps = {k: load(f, n) for k, f in
        dict(macro=F_MACRO, RUDY=F_RUDY, RUDY_pin=F_RPIN, overflow_H=F_LH, overflow_V=F_LV).items()}
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for a, (k, v) in zip(ax, maps.items()):
    im = a.imshow(v, cmap='inferno'); a.set_title(f'{k}\n{v.shape} [{v.min():.2g},{v.max():.2g}]'); a.axis('off')
    plt.colorbar(im, ax=a, fraction=0.046)
plt.show()

### Build the preprocessed dataset (my own pipeline, not the official script)
- resize every map to 256×256 (bilinear for continuous maps, nearest for the binary macro map)
- stack inputs → `X` of shape (3, 256, 256); labels → `Y` of shape (2, 256, 256)
- normalisation statistics computed **on the training designs only** (no test leakage)
- split by **design**, not by random sample

In [ ]:
import torch, torch.nn.functional as Fn, json
from tqdm.auto import tqdm

SPLIT = {  # design-held-out split
    'train': ['RISCY-a', 'RISCY-b', 'RISCY-FPU-a'],
    'val':   ['RISCY-FPU-b'],
    'test':  ['zero-riscy-a', 'zero-riscy-b'],
}
S = 256

def resize(a, mode):
    t = torch.from_numpy(a)[None, None]
    kw = {} if mode == 'nearest' else {'align_corners': False}
    return Fn.interpolate(t, size=(S, S), mode=mode, **kw)[0, 0].numpy()

def build(split, limit=None):
    idx = [i for i, d in enumerate(designs) if d in SPLIT[split]]
    if limit: idx = idx[:limit]
    X = np.zeros((len(idx), 3, S, S), np.float16); Y = np.zeros((len(idx), 2, S, S), np.float16)
    for j, i in enumerate(tqdm(idx, desc=split)):
        n = names[i]
        X[j, 0] = resize(load(F_MACRO, n), 'nearest')
        X[j, 1] = resize(load(F_RUDY, n), 'bilinear')
        X[j, 2] = resize(load(F_RPIN, n), 'bilinear')
        Y[j, 0] = resize(load(F_LH, n), 'bilinear')
        Y[j, 1] = resize(load(F_LV, n), 'bilinear')
    return X, Y

# Start small to confirm it works, then remove `limit`.
LIMIT = 200
data = {s: build(s, LIMIT) for s in SPLIT}

# Normalise with TRAIN statistics only (99.9th percentile clip -> [0,1])
Xtr, Ytr = data['train']
stats = {'x_hi': [float(np.percentile(Xtr[:, c], 99.9)) or 1.0 for c in range(3)],
         'y_hi': [float(np.percentile(Ytr[:, c], 99.9)) or 1.0 for c in range(2)]}
print(stats)
for s, (X, Y) in data.items():
    for c in range(3): X[:, c] = np.clip(X[:, c] / stats['x_hi'][c], 0, 1)
    for c in range(2): Y[:, c] = np.clip(Y[:, c] / stats['y_hi'][c], 0, 1)
    np.savez_compressed(f'{OUT}/{s}.npz', X=X, Y=Y)
    print(s, X.shape, Y.shape)
json.dump(stats, open(f'{OUT}/norm_stats.json', 'w'))

### Sanity checks for the proposal (numbers + a figure you can screenshot)

In [ ]:
for s, (X, Y) in data.items():
    hot = (Y.max(axis=1) > 0.5).mean()
    print(f'{s:5s}: n={len(X):5d}  mean RUDY={X[:,1].mean():.3f}  frac pixels with high overflow={hot:.3%}')

# Correlation between RUDY (the classic heuristic) and true overflow — motivates the baseline
r = np.corrcoef(data['train'][0][:, 1].ravel().astype(np.float32),
                data['train'][1].max(axis=1).ravel().astype(np.float32))[0, 1]
print('Pearson r(RUDY, overflow) on train subset:', round(r, 3))